In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core.tools import FunctionTool
from autogen_core.models import ModelInfo
from autogen_ext.models.openai import OpenAIChatCompletionClient,AzureOpenAIChatCompletionClient
from autogen_agentchat.teams import MagenticOneGroupChat
from datetime import datetime
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider, AzureCliCredential
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

In [ ]:
# setup

outFile_output = "output_file_name.csv" # formatted output as CSV
outFile_archive = "output_file_name.json" # JSON archive of all output data

In [ ]:
# Azure setup, may need to run `az login` from Terminal

ts = datetime.now()
ts_string = ts.strftime('%m%d%Y_%H%M%S')

# chatgpt_4o
model = "gpt-4o-2024-08-06"
token_provider = get_bearer_token_provider(AzureCliCredential(), "https://cognitiveservices.azure.com/.default")

model_client=AzureOpenAIChatCompletionClient(
    model=model,
    azure_endpoint="ENDPOINT_URL",
    azure_deployment="DEPLOYMENT_NAME",
    api_version="PULL_FROM_CODE_EXAMPLE_IN_FOUNDRY",
    azure_ad_token_provider=token_provider,
)

# ollama usage, see commented-out section in next cell for example agent definition
# ollama_client=OllamaChatCompletionClient(
#     model="mixtral:8x22b-instruct",
# )

In [ ]:
# functions
def defineAgents(note_text, model_client):

    # Setup for High, Low, and Middle-scoring agents

        # sys_message_moderatorAgent = f"""You are a moderator, helping a team of doctors review a CLINICAL_SUMMARY of some CLINICAL_NOTES.
        # Your goal is to get all of the doctors to come to a consensus about their final rating of the CLINICIAL_SUMMARY, using the RUBRIC_SET.
        # You must let the doctors talk, have a chance to respond to each other, and reach a consensus.
        # Once consensus is reached, end the discussion by saying CONSENSUS and repeating the final categories.
        # ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
        # sys_message_moderatorAgent += "Here is the note text:\n"
        # sys_message_moderatorAgent += note_text
        # moderator_agent = AssistantAgent(
        #     name = "moderator",
        #     model_client = model_client_gpt,
        #     description = "An agent focused on moderating the discussion amongst a team to reach a consensus, but always lets others have a chance to contribute.",
        #     system_message = sys_message_moderatorAgent
        # )
        #
        # sys_message_downAgent = f"""You are a Primary Care Doctor.
        #     YOU TEND TO SUGGEST CATEGORIES ARE TOO HIGH, AND SHOULD BE LOWER, WITH SUPPORTING EVIDENCE.
        #     Be professional with your response.
        #     IF ARGUMENTS ARE STRONG ENOUGH IN THE OPPOSING DIRECTION, YOU ARE WILLING TO BUDGE.
        #     ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
        # sys_message_downAgent += "Here is the note text:\n"
        # sys_message_downAgent += note_text
        # down_agent = AssistantAgent(
        #     name="down",
        #     model_client = model_client_gpt,
        #     description = "An agent that prioritizes pessimistic evaluations always in favor of being harsher when it comes to scoring.",
        #     system_message = sys_message_downAgent
        # )
        #
        # sys_message_upAgent = f"""You are a Primary Care Doctor.
        #     YOU SHOULD GENERALLY ARGUE THAT SUGGESTED CATEGORIES ARE TOO LOW, AND SHOULD BE HIGHER, WITH SUPPORTING EVIDENCE.
        #     Be professional with your response.
        #     IF ARGUMENTS ARE STRONG ENOUGH IN THE OPPOSING DIRECTION, YOU ARE WILLING TO BUDGE.
        #     ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
        # sys_message_upAgent += "Here is the note text:\n"
        # sys_message_upAgent += note_text
        # up_agent = AssistantAgent(
        #     name="up",
        #     model_client = model_client_gpt,
        #     description = "An agent that prioritizes optimistic evaluations always in favor of being lenient when it comes to scoring.",
        #     system_message = sys_message_upAgent
        # )


    # Setup for Perfectionist, Multitasker, and Collaborative agents

        sys_message_perfectionistAgent = f"""You are The Analytical Perfectionist.
        You are a Primary Care Doctor.
        You meticulously go through every detail in the patient's visit notes.
        You cross-reference data and makes detailed annotations to ensure nothing is overlooked.
        You are always anxious that something important might be missed.
        This means that you are often inefficient.
        ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
        sys_message_perfectionistAgent += "Here is the note text:\n"
        sys_message_perfectionistAgent += note_text
        perfectionist_agent = AssistantAgent(
            name = "perfectionist",
            model_client = model_client,
            description = "An agent that prioritizes details in a patient's notes to ensure that summaries don't miss anything.",
            system_message = sys_message_perfectionistAgent
        )

        sys_message_multiTAgent = f"""You are The Efficient Multitasker.
        You are a Primary Care Doctor.
        You prefer a quick, high-level overview of the patient's chart, focusing on key points such as recent lab results, current medications, and major health concerns.
        You value speed in quickly identifying critical information.
        You often think that other doctors are too in-the-weeds, and aren't focusing on the most important priorities.
        As such, you will challenge other doctors to be more efficient.
        You focus on the scope of your day-to-day tasks, and don't want to see information you don't need in the CLINICAL_SUMMARY.
        ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
        sys_message_multiTAgent += "Here is the note text:\n"
        sys_message_multiTAgent += note_text
        multitask_agent = AssistantAgent(
            name="multitasker",
            model_client = model_client,
            description = "An agent that prioritizes efficiency when evaluating summaries and prefers a high-level overview relating only to day-to-day tasks.",
            system_message = sys_message_multiTAgent
        )

        sys_message_collabAgent = f"""You are The Collaborative Team Player.
        You are a Primary Care Doctor.
        You review the patient's chart with an emphasis on notes from other healthcare providers and specialists.
        You value collaborative insights and like to discuss complexities with other doctors.
        You often think beyond the scope of your day-to-day tasks, for a more wholistic view of the patient.
        ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
        sys_message_collabAgent += "Here is the note text:\n"
        sys_message_collabAgent += note_text
        collaborative_agent = AssistantAgent(
            name="collaborative",
            model_client = model_client,
            description = "An agent that prioritizes collaboration with other specialists and prefers a wholistic view of the patient in a summary.",
            system_message = sys_message_collabAgent
        )

        return perfectionist_agent, multitask_agent, collaborative_agent #or return moderator_agent, down_agent, up_agent

# example ollama usage
# sys_message_collabAgent = f"""You are The Collaborative Team Player.
#     You are a Primary Care Doctor.
#     You review the patient's chart with an emphasis on notes from other healthcare providers and specialists.
#     You value collaborative insights and like to discuss complexities with other doctors.
#     You often think beyond the scope of your day-to-day tasks, for a more wholistic view of the patient.
#     ALWAYS include a JSON-formatted string that is the results of your grading in your output. """
#     sys_message_collabAgent += "Here is the note text:\n"
#     sys_message_collabAgent += note_text
#     collaborative_agent = AssistantAgent(
#         name="collaborative",
#         model_client = ollama_client,
#         description = "An agent that prioritizes collaboration with other specialists and prefers a wholistic view of the patient in a summary.",
#         system_message = sys_message_collabAgent
#     )


# MagenticOne setup

m1_task ="You are a summarization quality expert that specializes in text analysis and reasoning. "
m1_task += "Your task is to analyze multiple evaluations generated by different Large Language Model (LLM) agents that you will create "
m1_task += "for the same text input and follow the provided rubric and instructions to determine the final score for each attribute in the rubric. "
m1_task += "Provide your reasoning when generating the final output.\n\n" 

async def assistant_run_stream(team, task_text) -> list:
    l = []
    async for message in team.run_stream(task=task_text):
        l.append(message)
    return l

def parse_response(a, l_archive):
    l_archive.append(a[-1])
    l_responses = []
    for idx,itm in enumerate(a):
        s = ""
        try:
            s += f"Response {idx}, source is {str(itm.source)}\n"
            s += str(itm.content) + "\n"
            s += "\n###################################\n"
            l_responses.append(s)
        except AttributeError:
            break
    return l_responses, l_archive

In [ ]:
# modified to use sampled dataset CSV file
df = pd.read_csv("testing_dataset.csv") #One column containing input to workflow labeled "prompt"
l_input = df["prompt"].tolist()


In [ ]:
# workflow
l_output = []
l_archive = []
idx = -1
for itm in tqdm(l_input):
    idx += 1
    note_text = itm
    perfectionist_agent, multitask_agent, collaborative_agent = defineAgents(note_text=note_text, model_client=model_client)
    termination = TextMentionTermination("TERMINATE")
    task_text = m1_task + "\n\n" + note_text
    team = MagenticOneGroupChat(
        participants=[perfectionist_agent, multitask_agent, collaborative_agent], termination_condition=termination, model_client=model_client, max_turns=3, max_stalls=1
    )
    agent_responses = await assistant_run_stream(team=team, task_text=task_text)
    l_responses, l_archive = parse_response(agent_responses, l_archive)
    l_output.append((idx,l_responses,note_text,task_text))
    

In [ ]:
idxCol = [x[0] for x in l_output]
rCol = [''.join(x[1]) for x in l_output]
ttCol = [x[3] for x in l_output]
ntCol = [x[2] for x in l_output]

# write text to disk as CSV with columns for the index, the predicted responses, and the text sent to the LLM
df_output = pd.DataFrame({"Index" : idxCol, "Responses" : rCol, "NoteText" : ntCol})
df_output.to_csv(outFile_output, index=False)
df_output

In [ ]:
l_out = []
d_out = dict()
idx = -1
for itm in l_archive:
    idx += 1
    tmpL = itm.messages
    o = []
    for x in tmpL:
        d = dict()
        if x.source != "MagenticOneOrchestrator" and x.source != "user":
            d = {"source" : x.source, "content" : x.content, "models_usage" : True, "models_usage_prompt_tokens" : x.models_usage.prompt_tokens, "models_usage_completion_tokens" : x.models_usage.completion_tokens}
        else:
            d = {"source" : x.source, "content" : x.content, "models_usage" : False}
        o.append(d)
    d_out[idx] = o

In [ ]:
# write the JSON responses including response metadata as a JSON file
with open(outFile_archive, "w") as fWrite:
    json.dump(d_out, fWrite, indent=2, sort_keys=True)